# Read and write CSV with rugo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/csv.ipynb)

Rugo reads and writes CSV with no PyArrow, pandas or NumPy. Column types are inferred from a sample, or declared up front; columns and rows you don't ask for are dropped as the file is parsed.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.csv", "space_missions.csv")
print(f"space_missions.csv: {os.path.getsize('space_missions.csv'):,} bytes")

## Read the schema

`read_metadata` counts the rows and infers each column's type from the first rows. The first launches in this file have no price, so `Price` shows as text here; `read_csv` samples non-empty values instead, and reads it as a float.

In [ ]:
from rugo import csv

meta = csv.read_metadata("space_missions.csv")
print(meta.num_rows, "rows")
for column in meta.schema_columns:
    print(column)

## Read the whole file

`read_csv` is a context manager that yields a **morsel** - a batch of typed columns.

In [ ]:
with csv.read_csv("space_missions.csv") as reader:
    for morsel in reader:
        print(morsel)

## Declare types you already know

Inference picks numbers and text. The launch time is ISO-8601 text in the file; declare it with `explicit_schema` and it's parsed straight into a timestamp.

In [ ]:
with csv.read_csv("space_missions.csv", explicit_schema={"Lauched_at": "TIMESTAMP[us]"}) as reader:
    morsel = next(iter(reader))

for name, kind in zip(morsel.column_names, morsel.column_types):
    print(f"{name.decode():16} {kind}")

## Read only what you need

`columns` picks the columns; `predicates` filters rows as they're parsed. A predicate value must match the column's type - `Price` is a float, so compare it with a float.

In [ ]:
with csv.read_csv(
    "space_missions.csv",
    columns=["Company", "Mission", "Price"],
    predicates=[("Price", ">", 100.0)],
) as reader:
    expensive = next(iter(reader))

print(expensive.num_rows, "launches over $100M")
print(expensive)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list - only do this for the columns you need, since it's the step that creates a Python object per value.

In [ ]:
companies = expensive.column("Company").to_pylist()
print(sorted(set(companies)))

## Write CSV

`write_csv` turns a morsel into RFC 4180 CSV bytes. Pass `delimiter="\t"` for TSV, and read it back the same way.

In [ ]:
with open("expensive.tsv", "wb") as f:
    f.write(csv.write_csv(expensive, delimiter="\t"))

with csv.read_csv("expensive.tsv", delimiter="\t") as reader:
    print(next(iter(reader)).num_rows, "rows read back from the TSV")